# S48_05 — Multi-Agent Frameworks

Multi-agent systems use multiple specialized LLM agents that coordinate, delegate tasks, and check each other's work — similar to a software team with different roles.

## Why multi-agent?

- **Specialization**: a coding agent, a testing agent, a documentation agent each excel at one thing
- **Context management**: tasks too large for one context window can be split across agents
- **Parallelism**: independent subtasks can run concurrently
- **Verification**: one agent produces, another verifies — reduces errors

## Supervisor pattern

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# --- Specialist agents ---

def research_agent(topic: str) -> str:
    """Agent specialized in gathering factual information."""
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        system='You are a research specialist. Provide concise, factual summaries on topics. Include key statistics and dates.',
        messages=[{'role': 'user', 'content': f'Research this topic: {topic}'}],
    )
    return msg.content[0].text

def writer_agent(research: str, audience: str) -> str:
    """Agent specialized in writing clear explanations."""
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        system=f'You are a technical writer. Write clear, engaging explanations for {audience}.',
        messages=[{'role': 'user', 'content': f'Write an explanation based on this research:\n{research}'}],
    )
    return msg.content[0].text

def reviewer_agent(content: str) -> dict:
    """Agent specialized in quality review."""
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        system='You review content for accuracy and clarity. Respond with JSON only.',
        messages=[{'role': 'user', 'content': f'Review this content. Respond with {{"score": 1-10, "issues": [str], "approved": bool}}:\n{content}'}],
    )
    return json.loads(msg.content[0].text)

print('Specialist agents: research, writer, reviewer')

In [ ]:
# --- Supervisor agent ---

def supervisor_agent(task: str):
    """Orchestrates specialist agents to complete a writing task."""
    print(f'\n[Supervisor] Task: {task}')
    
    # Step 1: Research
    print('[Supervisor] → Dispatching to research agent...')
    research = research_agent(task)
    print(f'[Research] Done ({len(research)} chars)')
    
    # Step 2: Write
    print('[Supervisor] → Dispatching to writer agent...')
    article = writer_agent(research, audience='data science students')
    print(f'[Writer] Done ({len(article)} chars)')
    
    # Step 3: Review (with retry)
    for attempt in range(2):
        print(f'[Supervisor] → Review attempt {attempt + 1}...')
        review = reviewer_agent(article)
        print(f'[Reviewer] Score: {review["score"]}/10, Approved: {review["approved"]}')
        
        if review['approved']:
            break
        
        # Revise based on feedback
        if attempt == 0 and review['issues']:
            print(f'[Supervisor] Issues found: {review["issues"]}')
            article = writer_agent(
                research + f'\nPlease fix these issues: {review["issues"]}',
                audience='data science students'
            )
    
    return {'article': article, 'review': review}

result = supervisor_agent('attention mechanism in transformers')
print('\n--- Final article excerpt ---')
print(result['article'][:400] + '...')

## Multi-agent with LangGraph

In [ ]:
# LangGraph supervisor pattern — routes between specialist sub-agents
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages

class WorkflowState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str
    draft: str

llm = ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0)

def supervisor_node(state: WorkflowState):
    """Decides which agent to call next."""
    decision_msg = llm.invoke([
        SystemMessage(content='You route tasks. Reply with ONLY one word: "researcher", "writer", or "DONE".'),
        HumanMessage(content=f'Task: {state["messages"][0].content}\nDraft so far: {state.get("draft", "none")}'),
    ])
    next_agent = decision_msg.content.strip().lower()
    return {'next_agent': next_agent}

def researcher_node(state: WorkflowState):
    result = llm.invoke([
        SystemMessage(content='You are a researcher. Provide key facts.'),
        *state['messages'],
    ])
    return {'draft': f'Research: {result.content}', 'messages': [result]}

def writer_node(state: WorkflowState):
    result = llm.invoke([
        SystemMessage(content='You are a writer. Turn the research into a clear paragraph.'),
        HumanMessage(content=state.get('draft', '')),
    ])
    return {'draft': result.content, 'messages': [result]}

def route(state: WorkflowState) -> Literal['researcher', 'writer', '__end__']:
    next_a = state.get('next_agent', 'researcher')
    if next_a == 'done':
        return '__end__'
    return next_a if next_a in ['researcher', 'writer'] else '__end__'

workflow = StateGraph(WorkflowState)
workflow.add_node('supervisor', supervisor_node)
workflow.add_node('researcher', researcher_node)
workflow.add_node('writer', writer_node)

workflow.set_entry_point('supervisor')
workflow.add_conditional_edges('supervisor', route)
workflow.add_edge('researcher', 'supervisor')
workflow.add_edge('writer', 'supervisor')

multi_agent = workflow.compile()
print('Multi-agent graph compiled')

In [ ]:
result = multi_agent.invoke({
    'messages': [HumanMessage(content='Explain what RLHF is in 2 sentences.')],
    'next_agent': 'researcher',
    'draft': '',
})
print('Final draft:', result['draft'])

## Multi-agent framework landscape

| Framework | Approach | Best for |
|-----------|---------|----------|
| **LangGraph** | State graph, explicit control | Production agents, complex workflows |
| **CrewAI** | Role-based agents with tasks | Team simulation, easy setup |
| **AutoGen** (Microsoft) | Conversational agents | Research, code generation pairs |
| **Swarm** (OpenAI) | Lightweight agent handoffs | Simple routing patterns |
| **Claude Agent SDK** (Anthropic) | Subagent spawning via tool use | Parallel tasks, delegation |

> **2026 context:** Multi-agent systems are the dominant pattern for automating complex knowledge-work tasks. The key challenges are: reliability (agents failing mid-task), cost management, and human oversight. LangGraph is the most production-adopted framework as of 2026.

This completes S48. Next section: [S49_LLM_Finetuning](../06_LLM_Finetuning/S49_01_when_to_finetune.ipynb)